In [1]:
# 1) Imports and configuration
import os
import time
import random
import numpy as np
import torch
from torch.utils.data import IterableDataset, DataLoader
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import pandas as pd

data_path = "/home/pk222/projects/PDEControl_DPC/datasets/heat_smooth_f_dataset.npz"
samples_to_load = 100
seed = 32
train_frac = 0.8
batch_size = 75
num_steps = 1000
log_iter = 10
lr = 1e-3
transition_steps = 2000
decay_rate = 0.9
sample_id = 5
num_samples_to_evaluate = 10

resume_path = ""
resume_strict = True
save_best = True
save_last = True

# RNO preprocessing hyperparameters
sub = 1
T = 16
n_intervals = 5
train_up_to_tipping_point = False
tipping_data_split_prop = 0.67
debug_preprocessing_shapes = True


In [2]:
# 2) Reproducibility and device
random.seed(seed)
np.random.seed(seed)
torch.manual_seed(seed)
torch.cuda.manual_seed_all(seed)
torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = False

if not torch.cuda.is_available():
    raise RuntimeError("CUDA is required but not available.")
device = torch.device("cuda")
print(f"Using device: {device}")


Using device: cuda


In [3]:
# 3) Data loading and RNO preprocessing
if not os.path.exists(data_path):
    raise FileNotFoundError(f"Dataset not found: {data_path}")

dataset = np.load(data_path)
solutions = torch.from_numpy(dataset["solutions"][:samples_to_load]).float()
controls = torch.from_numpy(dataset["controls"][:samples_to_load]).float()
x_cord = torch.from_numpy(dataset["x"]).reshape(-1, 1).float()
dt = float(dataset["dt"])

print(f"Loaded {samples_to_load}/{dataset['solutions'].shape[0]} samples:")
print(f"solutions: {tuple(solutions.shape)}")
print(f"controls: {tuple(controls.shape)}")
print(f"x: {tuple(x_cord.shape)}")
print(f"dt: {dt}")

Loaded 100/3000 samples:
solutions: (100, 401, 100)
controls: (100, 400, 4)
x: (100, 1)
dt: 0.001


In [4]:
def round_down(num, divisor):
    return num - (num % divisor)


def chunk_time_axis(arr, chunk_size):
    # arr shape: (ntraj, nt, nfeat) -> (ntraj, n_chunks, chunk_size, nfeat)
    chunks = torch.split(arr, chunk_size, dim=1)
    return torch.stack(chunks, dim=1)

In [5]:
# 1) Align solution/control time lengths and crop to multiple of T
ntraj = min(solutions.shape[0], controls.shape[0])
nt_shared = min(solutions.shape[1] - 1, controls.shape[1])
nt_usable = round_down(nt_shared, T)

solutions = solutions[:ntraj, :nt_usable, :]
controls = controls[:ntraj, :nt_usable, :]

# 2) Optional spatial subsampling on solution/grid only
solutions = solutions[:, :, ::sub]
x_cord = x_cord[::sub]

# 3) Optional pre-tipping crop
if train_up_to_tipping_point:
    tipping_idx = round_down(int(tipping_data_split_prop * solutions.shape[1]), T)
    solutions = solutions[:, :tipping_idx, :]
    controls = controls[:, :tipping_idx, :]

ntraj, nt, nx = solutions.shape
nf = controls.shape[-1]
n_chunks = nt // T

print(f"After preprocessing:")
print(f"solutions: {tuple(solutions.shape)}")
print(f"controls: {tuple(controls.shape)}")
print(f"x: {tuple(x_cord.shape)}")
print(f"ntraj: {ntraj}, nt: {nt}, nx: {nx}, nf: {nf}, n_chunks: {n_chunks}")


After preprocessing:
solutions: (100, 400, 100)
controls: (100, 400, 4)
x: (100, 1)
ntraj: 100, nt: 400, nx: 100, nf: 4, n_chunks: 25


Train/test split by trajectory (same strategy as ks/RNO_KS.py)

In [6]:
# 4) Train/test split by trajectory (same strategy as ks/RNO_KS.py)
n_train_traj = int(train_frac * ntraj)
n_test_traj = ntraj - n_train_traj

sol_train = solutions[:n_train_traj]
sol_test = solutions[n_train_traj:]
ctrl_train = controls[:n_train_traj]
ctrl_test = controls[n_train_traj:]

In [7]:
print(f"Before chunking:")
print(f"sol_train: {tuple(sol_train.shape)}")
print(f"ctrl_train: {tuple(ctrl_train.shape)}")
print(f"sol_test: {tuple(sol_test.shape)}")
print(f"ctrl_test: {tuple(ctrl_test.shape)}")

Before chunking:
sol_train: (80, 400, 100)
ctrl_train: (80, 400, 4)
sol_test: (20, 400, 100)
ctrl_test: (20, 400, 4)


Chunk with T=16: (400 timesteps)

Raw time: 400
Chunk size: T=16
Number of chunks per trajectory: 400/16 = 25

U_chunks: (B, 25, 16, 100, 1)
F_chunks: (B, 25, 16, 4)

In [ ]:

# 5) Chunk both streams (keep state and control separate)
sol_train_chunks = chunk_time_axis(sol_train, T).unsqueeze(
    -1
)  # (n_tr, n_chunks, T, nx, 1)
sol_test_chunks = chunk_time_axis(sol_test, T).unsqueeze(-1)  # (n_te, n_chunks, T, nx, 1)

ctrl_train_chunks = chunk_time_axis(ctrl_train, T).unsqueeze(
    3
)  # (n_tr, n_chunks, T, 1, nf)
ctrl_test_chunks = chunk_time_axis(ctrl_test, T).unsqueeze(3)  # (n_te, n_chunks, T, 1, nf)

print("After chunking:")
print(f"sol_train_chunks: {tuple(sol_train_chunks.shape)}")
print(f"ctrl_train_chunks: {tuple(ctrl_train_chunks.shape)}")
print(f"sol_test_chunks: {tuple(sol_test_chunks.shape)}")
print(f"ctrl_test_chunks: {tuple(ctrl_test_chunks.shape)}")


Build training samples with a sliding window of `n_intervals=5`:

Input uses 5 consecutive chunks: `[i, i+1, i+2, i+3, i+4]`
Target is next chunk: `[i+5]`

So for the model input:

* `timesteps = 5` (history length in chunk units)
* `spatial_dims = (16, 100)` (chunk-internal time, space)
* That gives input like `(batch, 5, channels, 16, 100)` after permute.

Mini picture for one trajectory:

- chunks: `c0 c1 c2 c3 c4 c5 c6 ... c24`
- sample 1: `input=c0..c4`, `target=c5`
- sample 2: `input=c1..c5`, `target=c6`
etc.

In [ ]:

def build_windows(sol_chunks, ctrl_chunks, n_intervals_inner):
    x_state, x_ctrl, y_state = [], [], []
    for traj_sol, traj_ctrl in zip(sol_chunks, ctrl_chunks):
        for i in range(sol_chunks.shape[1] - n_intervals_inner - 1):
            x_state.append(traj_sol[i : i + n_intervals_inner])
            x_ctrl.append(traj_ctrl[i : i + n_intervals_inner])
            y_state.append(traj_sol[i + n_intervals_inner])

    x_state = torch.stack(x_state, dim=0)  # (N, n_intervals, T, nx, 1)
    x_ctrl = torch.stack(x_ctrl, dim=0)  # (N, n_intervals, T, 1, nf)
    y_state = torch.stack(y_state, dim=0)  # (N, T, nx, 1)

    x_state = x_state.permute(0, 1, 4, 2, 3).contiguous()  # (N, n_intervals, 1, T, nx)
    x_ctrl = x_ctrl.permute(0, 1, 3, 2, 4).contiguous()  # (N, n_intervals, 1, T, nf)
    y_state = y_state.permute(0, 3, 1, 2).contiguous()  # (N, 1, T, nx)
    return x_state, x_ctrl, y_state


x_train_state, x_train_ctrl, y_train = build_windows(
    sol_train_chunks, ctrl_train_chunks, n_intervals
)
x_test_state, x_test_ctrl, y_test = build_windows(
    sol_test_chunks, ctrl_test_chunks, n_intervals
)

if debug_preprocessing_shapes:
    print("=== RNO preprocessing debug ===")
    print(f"ntraj={ntraj}, nt={nt}, nx={nx}, nf={nf}, n_chunks={n_chunks}")
    print(f"n_train_traj={n_train_traj}, n_test_traj={n_test_traj}")
    print(f"sol_train_chunks: {tuple(sol_train_chunks.shape)}")
    print(f"ctrl_train_chunks: {tuple(ctrl_train_chunks.shape)}")
    print(f"x_train_state: {tuple(x_train_state.shape)}")
    print(f"x_train_ctrl: {tuple(x_train_ctrl.shape)}")
    print(f"y_train: {tuple(y_train.shape)}")
    print(f"x_test_state: {tuple(x_test_state.shape)}")
    print(f"x_test_ctrl: {tuple(x_test_ctrl.shape)}")
    print(f"y_test: {tuple(y_test.shape)}")
    print("=== End preprocessing debug ===")


In [ ]:

# Dataloaders for RNO training (state, control, target_state)
train_loader = DataLoader(
    torch.utils.data.TensorDataset(x_train_state, x_train_ctrl, y_train),
    batch_size=batch_size,
    shuffle=True,
    drop_last=True,
)
test_loader = DataLoader(
    torch.utils.data.TensorDataset(x_test_state, x_test_ctrl, y_test),
    batch_size=batch_size,
    shuffle=False,
    drop_last=True,
)

xb_state, xb_ctrl, yb = next(iter(train_loader))
print(f"train batch state: {tuple(xb_state.shape)}")
print(f"train batch control: {tuple(xb_ctrl.shape)}")
print(f"train batch target: {tuple(yb.shape)}")


In [11]:
# 4) RNO utilities
import operator
from functools import reduce
from tqdm.auto import tqdm


class LpLoss(object):
    def __init__(self, d=2, p=2, size_average=True, reduction=True):
        assert d > 0 and p > 0
        self.d = d
        self.p = p
        self.reduction = reduction
        self.size_average = size_average

    def rel(self, x, y):
        num_examples = x.size()[0]
        diff_norms = torch.norm(
            x.reshape(num_examples, -1) - y.reshape(num_examples, -1), self.p, 1
        )
        y_norms = torch.norm(y.reshape(num_examples, -1), self.p, 1)
        if self.reduction:
            if self.size_average:
                return torch.mean(diff_norms / y_norms)
            return torch.sum(diff_norms / y_norms)
        return diff_norms / y_norms

    def __call__(self, x, y):
        return self.rel(x, y)


def count_params(model):
    c = 0
    for p in list(model.parameters()):
        c += reduce(operator.mul, list(p.size()), 1)
    return c


From documentation: Input tensor `x_train` with shape `(batch, timesteps, in_channels, *spatial_dims)`, where `len(spatial_dims) == self.n_dim`. The channel dimension MUST be at index 2 and the time dimension MUST be at index 1.

Source: https://neuraloperator.github.io/dev/modules/generated/neuralop.models.RNO.html#neuralop.models.RNO


## Audit: current behavior

### Strategy 1 currently in notebook (before refactor)
- State chunks are built as `(n_tr, n_chunks, 16, 100, 1)`.
- Controls are chunked as `(n_tr, n_chunks, 16, 4)`, then broadcast over the `x` axis with `unsqueeze(3).expand(-1, -1, -1, nx, -1)` to `(n_tr, n_chunks, 16, 100, 4)`.
- State and control are concatenated channel-last with `torch.cat([x_sol, x_ctrl], dim=-1)` to form `(N, n_intervals, 16, 100, 5)`.

### Exact model input and channels
- `in_channels` is set from `x_train.shape[-1]`, so the model is configured with `in_channels=5` (expected), `out_channels=1`.
- Batches are passed to `model.predict` as `x_in = x_batch.permute(0, 1, 4, 2, 3)`.
- Exact forward input shape is `(B, n_intervals, 5, 16, 100)`.

### Training target and loss behavior
- Target is only the next chunk: `y_sol[i + n_intervals]` with shape `(B, 16, 100, 1)`.
- Training uses `model.predict(x_in, num_steps=1)[:, -1]`, then permutes to `(B, 16, 100, 1)`.
- Loss is one-step relative L2 (`LpLoss`) over the full chunk field, not sequence loss over multiple future steps.
- This is teacher-forced history input (fixed ground-truth context window), not free-run multi-step autoregressive training.

### Existing rollout path
- Rollout comparison is in cells 18-19 via `evaluate_rollout_steps`.
- It uses `model.predict(x_in, num_steps=h)` from a fixed input window.
- For `h > 1`, future controls are not injected at each predicted step; rollout proceeds from predicted states only.


In [ ]:

# 5) Strategy 2 control-conditioning helpers + RNO model
import math
import torch.nn as nn
from neuralop.models import RNO
from neuralop.layers.embeddings import SinusoidalEmbedding

model_input_mode = "sin_embed"  # preferred strategy: {"sin_embed", "broadcast"}


# Probe SinusoidalEmbedding signature/behavior on this environment
probe_embedding = SinusoidalEmbedding(
    in_channels=4,
    num_frequencies=13,
    embedding_type="transformer",
    max_positions=10000,
)
probe_x = torch.randn(2, 7, 4)
probe_y = probe_embedding(probe_x)
print("SinusoidalEmbedding probe:", tuple(probe_x.shape), "->", tuple(probe_y.shape))
print("Probe out_channels:", probe_embedding.out_channels)


def lift_controls_broadcast(x_ctrl, target_nx=100):
    # x_ctrl: (B, T_hist, 1, 16, 4) -> (B, T_hist, 4, 16, target_nx)
    if x_ctrl.ndim != 5:
        raise ValueError(f"Expected x_ctrl rank-5, got shape {tuple(x_ctrl.shape)}")
    if x_ctrl.shape[2] != 1:
        raise ValueError(f"Expected x_ctrl channel dim=1, got {x_ctrl.shape[2]}")

    ctrl = x_ctrl.squeeze(2)  # (B, T_hist, 16, 4)
    ctrl = ctrl.permute(0, 1, 3, 2)  # (B, T_hist, 4, 16)
    ctrl = ctrl.unsqueeze(-1)  # (B, T_hist, 4, 16, 1)
    ctrl = ctrl.expand(-1, -1, -1, -1, target_nx)  # (B, T_hist, 4, 16, target_nx)
    return ctrl


class ControlLifterSin(nn.Module):
    # Lift control last-dim 4 -> embed_dim using SinusoidalEmbedding.
    def __init__(
        self,
        control_dim=4,
        embed_dim=100,
        num_frequencies=None,
        embedding_type="transformer",
        max_positions=10000,
    ):
        super().__init__()
        self.control_dim = int(control_dim)
        self.embed_dim = int(embed_dim)

        if num_frequencies is None:
            num_frequencies = int(math.ceil(self.embed_dim / (2 * self.control_dim)))

        self.embedding = SinusoidalEmbedding(
            in_channels=self.control_dim,
            num_frequencies=num_frequencies,
            embedding_type=embedding_type,
            max_positions=max_positions,
        )
        self.embedding_dim = int(self.embedding.out_channels)
        if self.embedding_dim < self.embed_dim:
            raise ValueError(
                f"Embedding out_channels={self.embedding_dim} < embed_dim={self.embed_dim}"
            )

    def forward(self, x_ctrl):
        # x_ctrl: (B, T_hist, 1, 16, 4) -> (B, T_hist, 1, 16, 100)
        if x_ctrl.ndim != 5:
            raise ValueError(f"Expected x_ctrl rank-5, got shape {tuple(x_ctrl.shape)}")

        batch_size, timesteps, channels, inner_t, control_dim = x_ctrl.shape
        if control_dim != self.control_dim:
            raise ValueError(
                f"Expected control_dim={self.control_dim}, got x_ctrl.shape[-1]={control_dim}"
            )

        x_flat = x_ctrl.reshape(
            batch_size * timesteps * channels * inner_t, control_dim
        )  # (B*T_hist*1*16, 4)
        x_emb = self.embedding(x_flat)  # (B*T_hist*1*16, embedding_dim)
        x_emb = x_emb[..., : self.embed_dim]  # (B*T_hist*1*16, 100)
        x_lift = x_emb.reshape(
            batch_size, timesteps, channels, inner_t, self.embed_dim
        )  # (B, T_hist, 1, 16, 100)
        return x_lift


def build_rno_input(x_state, x_ctrl, mode="broadcast", control_lifter=None):
    # x_state: (B, T_hist, 1, 16, 100), x_ctrl: (B, T_hist, 1, 16, 4)
    if x_state.ndim != 5 or x_ctrl.ndim != 5:
        raise ValueError(
            f"Expected rank-5 tensors, got x_state={tuple(x_state.shape)}, x_ctrl={tuple(x_ctrl.shape)}"
        )
    if x_state.shape[:2] != x_ctrl.shape[:2] or x_state.shape[3] != x_ctrl.shape[3]:
        raise ValueError(
            "State/control batch/time/inner-time dims must match: "
            f"x_state={tuple(x_state.shape)}, x_ctrl={tuple(x_ctrl.shape)}"
        )

    if mode == "broadcast":
        x_ctrl_lifted = lift_controls_broadcast(
            x_ctrl, target_nx=x_state.shape[-1]
        )  # (B, T_hist, 4, 16, 100)
    elif mode == "sin_embed":
        if control_lifter is None:
            raise ValueError("mode='sin_embed' requires a control_lifter module")
        x_ctrl_lifted = control_lifter(x_ctrl)  # (B, T_hist, 1, 16, 100)
    else:
        raise ValueError(f"Unknown mode={mode}. Use 'broadcast' or 'sin_embed'.")

    x_in = torch.cat([x_state, x_ctrl_lifted], dim=2)  # (B, T_hist, C_in, 16, 100)
    return x_in


class RNOControlAR(nn.Module):
    def __init__(self, rno, mode="sin_embed", control_lifter=None):
        super().__init__()
        self.rno = rno
        self.mode = mode
        self.control_lifter = control_lifter

    def forward_one(self, u_t, ctrl_t, hidden_states=None, return_hidden_states=False):
        # u_t: (B,1,16,100) or (B,1,1,16,100)
        # ctrl_t: (B,1,16,4) or (B,1,1,16,4)
        if u_t.ndim == 4:
            u_t = u_t.unsqueeze(1)  # (B, 1, 1, 16, 100)
        if ctrl_t.ndim == 4:
            ctrl_t = ctrl_t.unsqueeze(1)  # (B, 1, 1, 16, 4)

        if u_t.ndim != 5 or u_t.shape[1] != 1:
            raise ValueError(f"u_t must be (B,1,1,16,100), got {tuple(u_t.shape)}")
        if ctrl_t.ndim != 5 or ctrl_t.shape[1] != 1:
            raise ValueError(f"ctrl_t must be (B,1,1,16,4), got {tuple(ctrl_t.shape)}")

        x_in = build_rno_input(
            u_t, ctrl_t, mode=self.mode, control_lifter=self.control_lifter
        )  # (B, 1, C_in, 16, 100)
        u_next, hidden_states = self.rno(
            x_in,
            init_hidden_states=hidden_states,
            return_hidden_states=True,
            keep_states_padded=True,
        )  # u_next: (B, 1, 16, 100)

        if return_hidden_states:
            return u_next, hidden_states
        return u_next

    def rollout(
        self,
        u0,
        ctrl_seq,
        steps,
        teacher_forcing_states=None,
        use_teacher_forcing=False,
    ):
        # u0: (B,1,16,100) or (B,1,1,16,100)
        # ctrl_seq: (B, steps, 1, 16, 4)
        # returns: (B, steps+1, 1, 16, 100)
        if u0.ndim == 5:
            if u0.shape[1] != 1:
                raise ValueError(f"u0 with rank-5 must have time dim=1, got {tuple(u0.shape)}")
            current_state = u0[:, 0]  # (B, 1, 16, 100)
        elif u0.ndim == 4:
            current_state = u0  # (B, 1, 16, 100)
        else:
            raise ValueError(f"u0 must be rank 4 or 5, got {tuple(u0.shape)}")

        if ctrl_seq.ndim != 5:
            raise ValueError(f"ctrl_seq must be rank-5, got {tuple(ctrl_seq.shape)}")
        if ctrl_seq.shape[1] < steps:
            raise ValueError(
                f"ctrl_seq has {ctrl_seq.shape[1]} steps but rollout requested {steps}"
            )

        if use_teacher_forcing:
            if teacher_forcing_states is None:
                raise ValueError("use_teacher_forcing=True requires teacher_forcing_states")
            if teacher_forcing_states.shape[1] < steps:
                raise ValueError(
                    "teacher_forcing_states must have at least 'steps' timesteps "
                    f"(got {teacher_forcing_states.shape[1]} < {steps})"
                )

        traj = [current_state]
        hidden_states = None

        for k in range(steps):
            if use_teacher_forcing and k > 0:
                current_state = teacher_forcing_states[:, k]  # (B, 1, 16, 100)

            ctrl_t = ctrl_seq[:, k]  # (B, 1, 16, 4)
            u_next, hidden_states = self.forward_one(
                current_state,
                ctrl_t,
                hidden_states=hidden_states,
                return_hidden_states=True,
            )  # u_next: (B, 1, 16, 100)
            traj.append(u_next)
            current_state = u_next

        traj = torch.stack(traj, dim=1)  # (B, steps+1, 1, 16, 100)
        return traj


modes1 = 20
modes2 = 20
width = 28
n_layers = 3
domain_padding = [0.1, 0]

if model_input_mode == "sin_embed":
    in_channels = 2
    control_lifter = ControlLifterSin(
        control_dim=4,
        embed_dim=100,
        num_frequencies=13,
        embedding_type="transformer",
        max_positions=10000,
    ).to(device)
elif model_input_mode == "broadcast":
    in_channels = 5
    control_lifter = None
else:
    raise ValueError(f"Unknown model_input_mode={model_input_mode}")

out_channels = 1
learning_rate = lr
weight_decay = 1e-4
scheduler_step = 50
scheduler_gamma = 0.5
epochs = 25
save_weights = True

model = RNO(
    n_modes=(modes1, modes2),
    hidden_channels=width,
    in_channels=in_channels,
    out_channels=out_channels,
    n_layers=n_layers,
    domain_padding=domain_padding,
).to(device)

ar_model = RNOControlAR(
    rno=model,
    mode=model_input_mode,
    control_lifter=control_lifter,
).to(device)

print(f"model_input_mode={model_input_mode}")
print(f"in_channels={in_channels}, out_channels={out_channels}")
print("Model parameters:", count_params(model))

opt_params = list(model.parameters())
if control_lifter is not None:
    opt_params += list(control_lifter.parameters())

optimizer = torch.optim.Adam(opt_params, lr=learning_rate, weight_decay=weight_decay)
scheduler = torch.optim.lr_scheduler.StepLR(
    optimizer, step_size=scheduler_step, gamma=scheduler_gamma
)
lploss = LpLoss(size_average=False)


In [ ]:

# 6) Shape sanity (non-invasive debug cell)
model.eval()
if control_lifter is not None:
    control_lifter.eval()

with torch.no_grad():
    xb_state, xb_ctrl, yb = next(iter(train_loader))
    xb_state = xb_state.to(device).float()  # (B, T_hist, 1, 16, 100)
    xb_ctrl = xb_ctrl.to(device).float()  # (B, T_hist, 1, 16, 4)
    yb = yb.to(device).float()  # (B, 1, 16, 100)

    x_in_dbg = build_rno_input(
        xb_state,
        xb_ctrl,
        mode=model_input_mode,
        control_lifter=control_lifter,
    )  # (B, T_hist, C_in, 16, 100)
    y_pred_dbg = model.predict(x_in_dbg, num_steps=1)[:, -1]  # (B, 1, 16, 100)

    print("dataset batch output:")
    print("  state:", tuple(xb_state.shape))
    print("  control:", tuple(xb_ctrl.shape))
    print("  target:", tuple(yb.shape))
    print("model input tensor:", tuple(x_in_dbg.shape))
    print("model output tensor:", tuple(y_pred_dbg.shape))
    print("loss inputs:", tuple(y_pred_dbg.shape), tuple(yb.shape))

model.train()
if control_lifter is not None:
    control_lifter.train()


In [ ]:

# 7) RNO training loop (control-aware input construction)
num_train_samples = x_train_state.shape[0]
num_test_samples = x_test_state.shape[0]
training_loss_history = []
test_loss_history = []

print("Begin RNO training:")
for ep in range(1, epochs + 1):
    model.train()
    if control_lifter is not None:
        control_lifter.train()

    t_start = time.time()
    train_l2 = 0.0

    for x_state_batch, x_ctrl_batch, y_batch in tqdm(train_loader, leave=False):
        x_state_batch = x_state_batch.to(device).float()  # (B, T_hist, 1, 16, 100)
        x_ctrl_batch = x_ctrl_batch.to(device).float()  # (B, T_hist, 1, 16, 4)
        y_batch = y_batch.to(device).float()  # (B, 1, 16, 100)

        x_in = build_rno_input(
            x_state_batch,
            x_ctrl_batch,
            mode=model_input_mode,
            control_lifter=control_lifter,
        )  # (B, T_hist, C_in, 16, 100)
        out = model.predict(x_in, num_steps=1)[:, -1]  # (B, 1, 16, 100)

        loss = lploss(out, y_batch)
        train_l2 += loss.item()

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
        scheduler.step()

    model.eval()
    if control_lifter is not None:
        control_lifter.eval()

    test_l2 = 0.0
    with torch.no_grad():
        for x_state_batch, x_ctrl_batch, y_batch in test_loader:
            x_state_batch = x_state_batch.to(device).float()
            x_ctrl_batch = x_ctrl_batch.to(device).float()
            y_batch = y_batch.to(device).float()

            x_in = build_rno_input(
                x_state_batch,
                x_ctrl_batch,
                mode=model_input_mode,
                control_lifter=control_lifter,
            )
            out = model.predict(x_in, num_steps=1)[:, -1]
            test_l2 += lploss(out, y_batch).item()

    train_l2 /= num_train_samples
    test_l2 /= num_test_samples
    training_loss_history.append(train_l2)
    test_loss_history.append(test_l2)

    print(
        f"Epoch {ep:03d}/{epochs} | time={time.time() - t_start:.2f}s | train_l2={train_l2:.4e} | test_l2={test_l2:.4e}"
    )


In [ ]:

# 8) Save model weights
if save_weights:
    result_dir = os.path.join(os.getcwd(), f"result_rno_{seed}")
    os.makedirs(result_dir, exist_ok=True)
    model_path = os.path.join(result_dir, "rno_model_last.pt")
    torch.save(
        {
            "model_state_dict": model.state_dict(),
            "control_lifter_state_dict": (
                control_lifter.state_dict() if control_lifter is not None else None
            ),
            "optimizer_state_dict": optimizer.state_dict(),
            "scheduler_state_dict": scheduler.state_dict(),
            "epochs": epochs,
            "seed": seed,
            "config": {
                "modes1": modes1,
                "modes2": modes2,
                "width": width,
                "model_input_mode": model_input_mode,
                "in_channels": in_channels,
                "out_channels": out_channels,
                "n_layers": n_layers,
                "domain_padding": domain_padding,
                "learning_rate": learning_rate,
                "weight_decay": weight_decay,
                "scheduler_step": scheduler_step,
                "scheduler_gamma": scheduler_gamma,
                "T": T,
                "n_intervals": n_intervals,
                "sub": sub,
            },
            "train_loss_history": training_loss_history,
            "test_loss_history": test_loss_history,
        },
        model_path,
    )
    print(f"Saved weights to: {model_path}")
else:
    print("save_weights=False, skipping model save.")


In [ ]:

# 9) Control-aware rollout example (uses exogenous controls at each step)
model.eval()
if control_lifter is not None:
    control_lifter.eval()

with torch.no_grad():
    xb_state, xb_ctrl, _ = next(iter(test_loader))
    xb_state = xb_state.to(device).float()  # (B, T_hist, 1, 16, 100)
    xb_ctrl = xb_ctrl.to(device).float()  # (B, T_hist, 1, 16, 4)

    u0 = xb_state[:, 0]  # (B, 1, 16, 100)
    ctrl_seq = xb_ctrl[:, :2]  # (B, 2, 1, 16, 4)
    rollout_pred = ar_model.rollout(u0=u0, ctrl_seq=ctrl_seq, steps=2)

    print("u0:", tuple(u0.shape))
    print("ctrl_seq:", tuple(ctrl_seq.shape))
    print("rollout_pred:", tuple(rollout_pred.shape))


In [ ]:

# 10) Unit-test-like shape checks + tiny numerical smoke test (1 batch)
model.eval()
if control_lifter is not None:
    control_lifter.eval()

xb_state, xb_ctrl, _ = next(iter(train_loader))
xb_state = xb_state.to(device).float()  # (B, T_hist, 1, 16, 100)
xb_ctrl = xb_ctrl.to(device).float()  # (B, T_hist, 1, 16, 4)

B, T_hist = xb_state.shape[0], xb_state.shape[1]

x_in_broadcast = build_rno_input(
    xb_state,
    xb_ctrl,
    mode="broadcast",
    control_lifter=control_lifter,
)  # (B, T_hist, 5, 16, 100)
assert x_in_broadcast.shape == (B, T_hist, 5, 16, 100), x_in_broadcast.shape

x_in_sin = build_rno_input(
    xb_state,
    xb_ctrl,
    mode="sin_embed",
    control_lifter=control_lifter,
)  # (B, T_hist, 2, 16, 100)
assert x_in_sin.shape == (B, T_hist, 2, 16, 100), x_in_sin.shape

u0_hist = xb_state[:, 0]  # (B, 1, 16, 100)
with torch.no_grad():
    y_pred_roll = ar_model.rollout(
        u0=u0_hist,
        ctrl_seq=xb_ctrl[:, :T_hist],
        steps=T_hist,
    )  # (B, T_hist+1, 1, 16, 100)
    y_pred = y_pred_roll[:, 1:]  # (B, T_hist, 1, 16, 100)
assert y_pred.shape == (B, T_hist, 1, 16, 100), y_pred.shape

u0 = xb_state[:, 0]  # (B, 1, 16, 100)
ctrl_two = xb_ctrl[:, :2]  # (B, 2, 1, 16, 4)
with torch.no_grad():
    rollout_out = ar_model.rollout(u0=u0, ctrl_seq=ctrl_two, steps=2)
assert rollout_out.shape == (B, 3, 1, 16, 100), rollout_out.shape

with torch.no_grad():
    rollout_tf = ar_model.rollout(
        u0=u0,
        ctrl_seq=ctrl_two,
        steps=2,
        teacher_forcing_states=xb_state[:, :2],
        use_teacher_forcing=True,
    )
assert rollout_tf.shape == (B, 3, 1, 16, 100), rollout_tf.shape

u0_rand = torch.randn(2, 1, 16, 100, device=device)
ctrl_rand = torch.randn(2, 2, 1, 16, 4, device=device)
with torch.no_grad():
    rollout_rand = ar_model.rollout(u0=u0_rand, ctrl_seq=ctrl_rand, steps=2)
assert torch.isfinite(rollout_rand).all(), "Rollout produced non-finite values"

print("All shape checks and rollout smoke tests passed.")
